# Lab 01 — Partitions and Parallelism

**Tujuan:** Menghubungkan partisi, task, core, dan paralelisme.

**Lingkungan:** Spark standalone (`spark-master:7077`), Jupyter, dua worker. Jalankan sel secara berurutan. Setiap notebook dapat dijalankan sendiri.

**Metode:** Prediksi → jalankan → amati `explain()`/Spark UI → ubah satu variabel → simpulkan.

In [14]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Spark Training - Lab 01")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)

Spark: 3.5.7
Master: spark://spark-master:7077


## 1. Create a predictable dataset

Argumen keempat `spark.range` adalah jumlah partisi awal.

In [15]:
df = spark.range(0, 1_000_000, 1, 4)
print("Partitions:", df.rdd.getNumPartitions())
df.show(5)

Py4JJavaError: An error occurred while calling o101.javaToPython.
: java.lang.IllegalStateException: Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(Unknown Source)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(Unknown Source)
java.base/java.lang.reflect.Constructor.newInstance(Unknown Source)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:182)
py

## 2. Change partitions

**Prediksi:** apakah `repartition` langsung memindahkan data? Perhatikan bahwa `getNumPartitions()` dapat meminta analisis plan, tetapi bukan materialisasi penuh data.

In [10]:
four = df.repartition(4)
eight = df.repartition(8)
print("4:", four.rdd.getNumPartitions())
print("8:", eight.rdd.getNumPartitions())

[Stage 1:==============>                                            (1 + 3) / 4]

4: 4
8: 8


26/11/10 00:34:41 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: Master removed our application: KILLED
26/11/10 00:34:42 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exiting due to error from cluster scheduler: Master removed our application: KILLED
	at org.apache.spark.errors.SparkCoreErrors$.clusterSchedulerError(SparkCoreErrors.scala:291)
	at org.apache.spark.scheduler.TaskSchedulerImpl.error(TaskSchedulerImpl.scala:981)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend.dead(StandaloneSchedulerBackend.scala:165)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint.markDead(StandaloneAppClient.scala:263)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint$$anonfun$receive$1.applyOrElse(StandaloneAppClient.scala:170)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:115)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:213)
	at org.apache.spark.rpc.netty.Inbox.proce

In [4]:
eight.explain("formatted")
print("Rows:", eight.count())

== Physical Plan ==
AdaptiveSparkPlan (5)
+- == Final Plan ==
   ShuffleQueryStage (3), Statistics(sizeInBytes=15.3 MiB, rowCount=1.00E+6)
   +- Exchange (2)
      +- * Range (1)
+- == Initial Plan ==
   Exchange (4)
   +- Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 1000000, step=1, splits=Some(4))

(2) Exchange
Input [1]: [id#0L]
Arguments: RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=43]

(3) ShuffleQueryStage
Output [1]: [id#0L]
Arguments: 0

(4) Exchange
Input [1]: [id#0L]
Arguments: RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=38]

(5) AdaptiveSparkPlan
Output [1]: [id#0L]
Arguments: isFinalPlan=true


Rows: 1000000


## 3. Compare `coalesce` vs `repartition`

Perhatikan operator `Exchange` dan shuffle pada physical plan.

In [5]:
coalesced = eight.coalesce(2)
repartitioned = eight.repartition(2)
print("COALESCE")
coalesced.explain("formatted")
print("REPARTITION")
repartitioned.explain("formatted")

COALESCE
== Physical Plan ==
AdaptiveSparkPlan (4)
+- Coalesce (3)
   +- Exchange (2)
      +- Range (1)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 1000000, step=1, splits=Some(4))

(2) Exchange
Input [1]: [id#0L]
Arguments: RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=118]

(3) Coalesce
Input [1]: [id#0L]
Arguments: 2

(4) AdaptiveSparkPlan
Output [1]: [id#0L]
Arguments: isFinalPlan=false


REPARTITION
== Physical Plan ==
AdaptiveSparkPlan (3)
+- Exchange (2)
   +- Range (1)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 1000000, step=1, splits=Some(4))

(2) Exchange
Input [1]: [id#0L]
Arguments: RoundRobinPartitioning(2), REPARTITION_BY_NUM, [plan_id=127]

(3) AdaptiveSparkPlan
Output [1]: [id#0L]
Arguments: isFinalPlan=false




**Tugas:** catat jumlah task tiap stage; jelaskan mengapa 8 partisi ≠ 8 task yang pasti berjalan serentak.

## Refleksi peserta

Tuliskan: **(1)** apa prediksi Anda, **(2)** apa yang teramati di Spark UI atau output, **(3)** mengapa Spark berperilaku demikian. Hindari menyimpulkan dari waktu eksekusi satu kali saja.

In [6]:
spark.stop()